# Anomaly Detection Analysis

Explore anomaly-tagged records from the Silver and Gold layers.
Covers per-field anomaly breakdown, z-score distributions, device health, and quality scores.

In [ ]:
from pipeline.common.utils import load_config, get_spark_session

config = load_config()
spark = get_spark_session(config)

In [ ]:
silver_df = spark.read.format("delta").load(config["paths"]["delta_silver"])

anomalies = silver_df.filter(silver_df["_is_anomaly"] == True)
normal = silver_df.filter(silver_df["_is_anomaly"] == False)

total = silver_df.count()
anom_count = anomalies.count()
norm_count = normal.count()

print(f"Total records:   {total}")
print(f"Anomalies:       {anom_count} ({anom_count/total*100:.1f}%)")
print(f"Normal:          {norm_count}")

print("\nAnomaly type breakdown:")
silver_df.selectExpr(
    "sum(cast(_is_temp_anomaly as int)) as temperature",
    "sum(cast(_is_humidity_anomaly as int)) as humidity",
    "sum(cast(_is_pressure_anomaly as int)) as pressure",
).show(truncate=False)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pdf = anomalies.select(
    "device_id", "timestamp", "temperature", "humidity", "pressure",
    "_anomaly_details", "_quality_score",
).toPandas()

if not pdf.empty:
    fig = px.scatter(
        pdf, x="temperature", y="humidity",
        color="device_id", hover_data=["_anomaly_details"],
        title="Anomalous Readings: Temperature vs Humidity",
    )
    fig.show()
else:
    print("No anomalies to plot.")

### Z-Score Distributions

In [ ]:
zscore_pdf = silver_df.select(
    "_temperature_zscore", "_humidity_zscore", "_pressure_zscore", "_is_anomaly",
).toPandas()

fig = make_subplots(rows=1, cols=3, subplot_titles=["Temperature", "Humidity", "Pressure"])
for i, col_name in enumerate(["_temperature_zscore", "_humidity_zscore", "_pressure_zscore"], 1):
    clean = zscore_pdf[zscore_pdf["_is_anomaly"] == False][col_name].dropna()
    anom = zscore_pdf[zscore_pdf["_is_anomaly"] == True][col_name].dropna()
    fig.add_trace(go.Histogram(x=clean, name="Clean", marker_color="#2ecc71",
                               opacity=0.7, showlegend=(i == 1)), row=1, col=i)
    fig.add_trace(go.Histogram(x=anom, name="Anomalous", marker_color="#e74c3c",
                               opacity=0.7, showlegend=(i == 1)), row=1, col=i)
fig.update_layout(title="Z-Score Distributions (Clean vs Anomalous)",
                  barmode="overlay", template="plotly_white", height=350)
fig.show()

### Device Health from Gold Layer

In [ ]:
from pathlib import Path

health_path = str(Path(config["paths"]["delta_gold"]) / "device_health")
health_df = spark.read.format("delta").load(health_path)
health_pdf = health_df.select(
    "device_id", "health_score", "risk_tier", "anomaly_rate", "avg_battery_level",
).toPandas()

tier_colors = {"healthy": "#2ecc71", "warning": "#f39c12", "critical": "#e74c3c"}

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Health Score Distribution", "Risk Tier Breakdown"],
    specs=[[{"type": "histogram"}, {"type": "pie"}]])

for tier, color in tier_colors.items():
    subset = health_pdf[health_pdf["risk_tier"] == tier]
    if not subset.empty:
        fig.add_trace(go.Histogram(x=subset["health_score"], name=tier.capitalize(),
                                   marker_color=color, nbinsx=15), row=1, col=1)

tier_counts = health_pdf["risk_tier"].value_counts()
fig.add_trace(go.Pie(labels=[t.capitalize() for t in tier_counts.index],
                      values=tier_counts.values,
                      marker_colors=[tier_colors.get(t, "#888") for t in tier_counts.index],
                      hole=0.4), row=1, col=2)

fig.update_layout(title="Device Health Overview", template="plotly_white",
                  height=400, barmode="stack")
fig.show()